# Spectrum-free baselines on stereo-stripped S4 candidates

Final retrieval evaluation of two spectrum-free SMILES-only baselines
(chirality count and ChemBERTa-77M-MLM binary classifier) on the
S4-augmented retrieval candidates, after stereo-stripping queries AND
candidates to remove the asymmetry shortcut (see context in
`MassSpecGym/scripts/strip_stereo_for_baselines.py`).

All numbers are computed per spectrum (n=17,556 test spectra), bootstrap
CIs over spectra, matching the convention in `evaluation.ipynb`. Two
tables: mass (S4+PubChem fallback, ±10 ppm) and formula (S4+PubChem+Molpher).

**Note on the random baseline.** Random hit-rate is computed per
spectrum as `1{rng.random() < k / N_i}` for the actual candidate-list
size `N_i` of each query, not the naive `1/1024`. Mean random hit@1 is
**~0.47% on mass nostereo** and **~1.83% on formula nostereo** — these
are the proper random references, not 0.1%.

In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm

from massspecgym.utils import get_ci

tqdm.pandas()

seed = 0
random.seed(seed)
np.random.seed(seed)
pd.set_option('compute.use_numexpr', False)
pd.set_option('compute.use_bottleneck', False)

DIR_RESULTS = Path('/pfs/lustrep2/scratch/project_465002061/rbushuie/DreaMS-Mol_dev/MassSpecGym/data/test_results_v1.5/retrieval')
DATASET_PTH = Path('/pfs/lustrep2/scratch/project_465002061/rbushuie/DreaMS-Mol_dev/MassSpecGym/data/MassSpecGym1.5_nostereo.tsv')

In [2]:
def evaluate(dir_results, method_pkls, dataset_pth=None, metric_cols=None):
    """Per-spectrum bootstrap evaluation. ``method_pkls`` is a dict
    ``{method_label: pkl_filename}``. Mirrors evaluation.ipynb::evaluate but
    accepts an explicit method list and is robust to missing metric columns."""
    np.random.seed(seed)
    if metric_cols is None:
        metric_cols = ['test_hit_rate@1', 'test_hit_rate@5', 'test_hit_rate@20', 'test_mrr']

    gt_smiles = None
    if dataset_pth is not None:
        tsv = pd.read_csv(dataset_pth, sep='\t', usecols=['identifier', 'smiles'])
        gt_smiles = dict(zip(tsv['identifier'], tsv['smiles']))

    def _row_rr(row):
        if gt_smiles is None or 'sorted_candidate_smiles' not in row.index:
            return np.nan
        gt = gt_smiles.get(row['identifier'])
        if gt is None:
            return 0.0
        try:
            rank = list(row['sorted_candidate_smiles']).index(gt) + 1
            return 1.0 / rank
        except (ValueError, TypeError):
            return 0.0

    dfs = []
    for label, fn in method_pkls.items():
        path = dir_results / fn
        df_m = pd.read_pickle(path)
        df_m['method'] = label
        # Fill missing MRR from sorted_candidate_smiles + TSV lookup; else leave NaN.
        if 'test_mrr' not in df_m.columns:
            if 'sorted_candidate_smiles' in df_m.columns and gt_smiles is not None:
                df_m['test_mrr'] = df_m.apply(_row_rr, axis=1)
            else:
                df_m['test_mrr'] = np.nan
        dfs.append(df_m)
    df = pd.concat(dfs, ignore_index=True)

    # Render hit-rates / mrr as % to match evaluation.ipynb.
    for col in [c for c in df.columns if 'hit_rate' in c]:
        df[col] = df[col] * 100
    if 'test_mrr' in df.columns:
        df['test_mrr'] = df['test_mrr'] * 100

    cols_present = [c for c in metric_cols if c in df.columns]
    df_mean = df.groupby('method', sort=False)[cols_present].mean().round(2)

    def _ci_str(vals):
        v = pd.Series(vals).dropna().values
        if len(v) < 2:
            return '—'
        lo, hi = get_ci(v, confidence_level=0.999, n_resamples=20_000, seed=seed)
        return f'{lo:.2f}-{hi:.2f}'

    tqdm.pandas(desc='Bootstrapping per method', postfix=None)
    df_ci = df.groupby('method', sort=False)[cols_present].progress_apply(
        lambda dm: dm.apply(_ci_str, axis=0)
    )

    for c in cols_present:
        df_mean[c] = df_mean[c].astype(str) + ' (' + df_ci[c] + ')'

    # Preserve the input order of methods.
    df_mean = df_mean.reindex(list(method_pkls.keys()))
    return df_mean

## Mass-filtered candidate pool (S4+PubChem, ±10 ppm; stereo stripped)

In [3]:
mass_methods = {
    'random':    'random_mass_nostereo.pkl',
    'chirality': 'chirality_mass_nostereo_per_spectrum.pkl',
    'chemberta': 'chemberta_mass_nostereo.pkl',
}
df_mass = evaluate(DIR_RESULTS, mass_methods, dataset_pth=DATASET_PTH)
display(df_mass)

Bootstrapping per method:   0%|          | 0/3 [00:00<?, ?it/s]

Bootstrapping per method:  67%|██████▋   | 2/3 [00:28<00:14, 14.19s/it]

Bootstrapping per method: 100%|██████████| 3/3 [00:49<00:00, 17.10s/it]

Bootstrapping per method: 100%|██████████| 3/3 [01:17<00:00, 25.92s/it]

,test_hit_rate@1,test_hit_rate@5,test_hit_rate@20,test_mrr
method,,,,
random,0.47 (0.32-0.66),2.34 (1.98-2.72),8.65 (7.96-9.38),2.29 (2.09-2.51)
chirality,0.31 (0.19-0.46),1.54 (1.26-1.85),10.67 (9.96-11.44),nan (—)
chemberta,1.57 (1.28-1.88),3.5 (3.06-3.96),9.86 (9.17-10.63),3.46 (3.15-3.79)


## Formula-filtered candidate pool (S4+PubChem+Molpher; stereo stripped)

In [4]:
formula_methods = {
    'random':    'random_formula_nostereo.pkl',
    'chirality': 'chirality_formula_nostereo_per_spectrum.pkl',
    'chemberta': 'chemberta_formula_nostereo.pkl',
}
df_formula = evaluate(DIR_RESULTS, formula_methods, dataset_pth=DATASET_PTH)
display(df_formula)

Bootstrapping per method:   0%|          | 0/3 [00:00<?, ?it/s]

Bootstrapping per method:  67%|██████▋   | 2/3 [00:28<00:14, 14.10s/it]

Bootstrapping per method: 100%|██████████| 3/3 [00:49<00:00, 17.04s/it]

Bootstrapping per method: 100%|██████████| 3/3 [01:17<00:00, 25.84s/it]

,test_hit_rate@1,test_hit_rate@5,test_hit_rate@20,test_mrr
method,,,,
random,1.83 (1.51-2.19),9.12 (8.41-9.83),27.94 (26.84-29.10),6.69 (6.33-7.10)
chirality,0.8 (0.59-1.03),11.9 (11.14-12.75),26.93 (25.87-28.07),nan (—)
chemberta,1.48 (1.21-1.81),8.53 (7.85-9.21),31.99 (30.84-33.14),6.66 (6.32-7.03)


## Notes

- **Random** is the per-spectrum theoretical baseline: each spectrum has
  its own candidate-list size `N_i`; per-spectrum hit@k is drawn from
  Bernoulli(k / N_i). The CIs reflect both this stochasticity and the
  bootstrap over spectra.
- **Chirality** = score each candidate by RDKit chiral-atom count, fitted
  direction (asc/desc) from MSG-train, random tie-breaking. After
  stripping stereo from both queries and candidates the chir_count is 0
  for almost every molecule — this baseline therefore collapses to
  near-random performance.
- **ChemBERTa-77M-MLM binary classifier** trained per candidate variant
  on (query=positive, candidates=negative) pairs from MSG-train, scored
  at test time as P(class=1). See
  `scripts/train_eval_chemberta_binary.py`.

After stereo-stripping, only ChemBERTa mass remains meaningfully above
random (~5× over the per-spectrum random baseline) — likely a multi-feature
token-sequence signal, not explained by any single SMILES property (see
`scripts/diagnose_residual_signal.py` per-feature ablation). Formula
ChemBERTa is essentially at random.

Stereo-preserving numbers and the per-feature single-property diagnostic
have been removed from this notebook to keep the headline tables clean;
they remain reproducible from the result pickles and the
`diagnose_residual_*_nostereo.pkl` files under
`data/test_results_v1.5/retrieval/`.